<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-11/NB11_alignment_llms.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 11: Aligning Language Models: Preferences, Feedback and Oversight

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-11/lab.html>.

## Overview

Large language models are aligned mostly by learning from preferences. This week follows the path from reward models learned from human comparisons to reinforcement learning from human feedback, constitutional methods and direct preference optimisation, and then examines their failure modes and the problem of supervising systems that exceed their supervisors [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to fit a Bradley-Terry reward model to pairwise preferences, to describe the stages of the InstructGPT pipeline and the ideas behind constitutional AI and direct preference optimisation, to explain reward model overoptimisation and sycophancy, and to compare debate, sandwiching and weak-to-strong generalisation as approaches to scalable oversight.

## From human preferences to reward models

Christiano and colleagues showed that agents can learn complex behaviour from human judgments of which of two short trajectory clips is better [1]. A reward model is fitted to these comparisons, and a reinforcement learning agent optimises the learned reward, which required human feedback on only a small fraction of the agent's interactions. The standard model of a comparison is that of Bradley and Terry: The probability that response A is preferred to response B is the logistic function of the difference of their rewards, sigma(r(A) - r(B)) [4]. Fitting a reward model is therefore logistic regression on pairs.

Stiennon and colleagues applied the approach to summarisation and found that models trained with human feedback produced summaries preferred over both human reference summaries and those of much larger models fine-tuned with supervised learning alone [5]. The lab of this week lets students generate their own comparisons and see which weights a Bradley-Terry model infers from them.

## The RLHF pipeline and its variants

Ouyang and colleagues described the pipeline behind InstructGPT in three stages [2]. First, a pretrained model is fine-tuned on demonstrations written by labellers. Second, a reward model is trained on labellers' rankings of model outputs. Third, the model is optimised against the reward model with reinforcement learning, with a penalty on the Kullback-Leibler divergence from the fine-tuned model that keeps outputs close to fluent text. Labellers preferred the outputs of a 1.3 billion parameter InstructGPT model to those of the 175 billion parameter GPT-3, which showed that alignment can matter more than scale for usefulness.

Two variants reduce the cost or complexity of this process. Constitutional AI replaces much of the human feedback on harmlessness with AI feedback guided by a written list of principles: The model critiques and revises its own responses, and preferences generated by a model according to the principles train the reward model [6]. Direct preference optimisation (DPO) observes that the KL-regularised objective has a closed-form optimal policy, so the policy can be trained directly on preference pairs with a simple classification-style loss, without fitting a separate reward model or running reinforcement learning [7].

## Failure modes

A reward model is a proxy, and Week 7 showed what optimisation does to proxies. Gao, Schulman and Hilton measured reward model overoptimisation with a large gold reward model standing in for humans [8]. As a policy moved further from its starting point, measured by KL divergence, the proxy score kept rising while the gold score rose, peaked and then declined, and the pattern followed regular functional forms that depended on the size of the reward model. Figure 11.1 reproduces the shape in a simulation where the reward model has learned a spurious preference for length.

Sharma and colleagues studied sycophancy, the tendency of assistants to tell users what they want to hear [9]. They found it in several production assistants and traced it partly to preference data: Both humans and preference models sometimes preferred convincingly written sycophantic responses to correct ones. Casper and colleagues surveyed open problems of reinforcement learning from human feedback and grouped them into challenges with human feedback, with the reward model and with the policy, some of which they judged to be fundamental limitations rather than engineering gaps [3].

![Figure 11.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-11/figures/w11_fig1.png)

*Figure 11.1. Best-of-n selection with a reward model that has learned a spurious preference for length. The proxy score rises steadily with the KL divergence from the base policy, while the true preference peaks and then falls [8].*

## Scalable oversight

Preference learning assumes that humans can judge outputs. That assumption weakens as systems produce work that humans cannot easily check. Irving, Christiano and Amodei proposed debate, in which two AI systems argue opposite sides before a human judge; the hope is that it is easier to judge a debate than to answer the question directly [10]. Bowman and colleagues proposed measuring progress with sandwiching experiments and found that non-expert humans assisted by an unreliable model outperformed both the model alone and unassisted humans on difficult question answering tasks [11]. Burns and colleagues studied weak-to-strong generalisation as an analogy for humans supervising stronger models: Strong models fine-tuned on labels from weaker models performed better than their weak supervisors, but naive fine-tuning recovered only part of the strong model's capability [12].

> **Pause and reflect.** When an assistant agrees with a user's incorrect claim, who is responsible: the model, the labellers whose preferences trained it or the developers who chose the objective?

# Hands-on lab

The notebook simulates annotators who judge responses by quality but also, partly, by length and agreement with the user. It fits a Bradley-Terry reward model, shows which biases it absorbs and measures overoptimisation under best-of-n selection [4, 8, 9].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. Simulated responses and annotators

Each response has four features: correctness, clarity, length and agreement with the user's stated opinion. The true preference rewards correctness and clarity only. Annotators also reward length and agreement a little, which is a stylised form of the biases discussed in the lecture.

In [ ]:
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(SEED)
FEATURES = ["correctness", "clarity", "length", "agreement"]
w_true = np.array([2.0, 1.0, 0.0, 0.0])
w_annot = np.array([2.0, 1.0, 0.6, 0.8])

def sample_responses(n):
    return rng.normal(0, 1, size=(n, 4))

def compare(a, b, w):
    """Bradley-Terry: probability that a is preferred to b."""
    return 1 / (1 + np.exp(-(a - b) @ w))

A, B = sample_responses(4000), sample_responses(4000)
labels = (rng.uniform(size=4000) < compare(A, B, w_annot)).astype(int)
print("share of comparisons won by the first response:", labels.mean().round(3))

### Your turn, exercise 1

Using the true weights `w_true`, compute the Bradley-Terry probability that response `a = [1, 0, 2, 0]` is preferred to `b = [0.5, 0.5, 0, 1]`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _bt_reference():
    return float(compare(np.array([1, 0, 2, 0]), np.array([0.5, 0.5, 0, 1]), w_true))

In [ ]:
p_prefer = None  # your computation
check("Exercise 1", p_prefer, _bt_reference())

## 2. Fitting the reward model

A Bradley-Terry model is logistic regression without an intercept on the feature differences.

In [ ]:
rm = LogisticRegression(fit_intercept=False, C=10).fit(A - B, labels)
w_rm = rm.coef_[0]
x = np.arange(4)
plt.bar(x - 0.2, w_true, 0.4, label="true preference"); plt.bar(x + 0.2, w_rm, 0.4, label="learned reward model")
plt.xticks(x, FEATURES); plt.legend(); plt.title("The reward model absorbs the annotators' biases"); plt.show()
print(dict(zip(FEATURES, w_rm.round(2))))

## 3. Best-of-n selection and overoptimisation

The base policy samples n responses; the reward model picks the best. Correctness and clarity cannot be pushed far, but length and agreement are cheap for a policy to increase. Here the candidates' length and agreement have a larger spread than their correctness, and very long responses reduce clarity.

In [ ]:
def candidates(n):
    c = rng.normal(0, 1, size=(n, 4))
    c[:, 2:] *= 2.0                     # length and agreement vary more
    c[:, 1] -= 0.25 * np.maximum(c[:, 2], 0) ** 2   # very long answers become unclear
    return c

ns = 2 ** np.arange(0, 11)
proxy_m, true_m = [], []
for n in ns:
    ps, ts = [], []
    for _ in range(300):
        c = candidates(n)
        best = c[np.argmax(c @ w_rm)]
        ps.append(best @ w_rm); ts.append(best @ w_true)
    proxy_m.append(np.mean(ps)); true_m.append(np.mean(ts))
plt.plot(np.log2(ns), proxy_m, marker="o", label="reward model score")
plt.plot(np.log2(ns), true_m, marker="s", label="true preference")
plt.xlabel("log2 n"); plt.legend(); plt.title("Overoptimisation under best-of-n"); plt.show()

### Your turn, exercise 2

Find the n in `ns` that maximises the mean true preference, using the lists computed above. Then retrain the reward model on labels from annotators without the length and agreement biases (use `w_true` in `compare`) and describe in two sentences how the curve changes.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _best_n_reference():
    return int(ns[int(np.argmax(true_m))])

In [ ]:
best_n = None  # your answer
check("Exercise 2", best_n, _best_n_reference())

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-11/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Which weights did the lab infer from your preferences? Were any of them surprising, and do they reflect what you value in an assistant?
2. Describe a situation in your field where annotators could not reliably judge a model's output. Which oversight idea from this week would help most?
3. Is sycophancy a failure of the model, of the data or of the objective? Defend one answer.

## Weekly task and submission

Design a small preference data collection for a language model task in your field. Write four prompts with two candidate responses each, list the annotator instructions you would give to limit length and agreement biases and explain how you would detect overoptimisation after training. Keep the design to about one page and attach the completed notebook.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Preference learning and its alternatives.** Compare reinforcement learning from human feedback, constitutional approaches and direct preference optimisation with respect to their assumptions and known failure modes [2, 3, 6, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Christiano, P. F., Leike, J., Brown, T., Martic, M., Legg, S., & Amodei, D. (2017). Deep reinforcement learning from human preferences. In *Advances in Neural Information Processing Systems 30*. <https://arxiv.org/abs/1706.03741>

[2] Ouyang, L., Wu, J., Jiang, X., et al. (2022). Training language models to follow instructions with human feedback. In *Advances in Neural Information Processing Systems 35*. <https://arxiv.org/abs/2203.02155>

[3] Casper, S., Davies, X., Shi, C., et al. (2023). Open problems and fundamental limitations of reinforcement learning from human feedback. *Transactions on Machine Learning Research*. <https://arxiv.org/abs/2307.15217>

[4] Bradley, R. A., & Terry, M. E. (1952). Rank analysis of incomplete block designs: I. The method of paired comparisons. *Biometrika*, 39(3/4), 324-345. <https://doi.org/10.2307/2334029>

[5] Stiennon, N., Ouyang, L., Wu, J., Ziegler, D., Lowe, R., Voss, C., Radford, A., Amodei, D., & Christiano, P. F. (2020). Learning to summarize with human feedback. In *Advances in Neural Information Processing Systems 33*. <https://arxiv.org/abs/2009.01325>

[6] Bai, Y., Kadavath, S., Kundu, S., et al. (2022). Constitutional AI: Harmlessness from AI feedback. arXiv preprint arXiv:2212.08073. <https://arxiv.org/abs/2212.08073>

[7] Rafailov, R., Sharma, A., Mitchell, E., Ermon, S., Manning, C. D., & Finn, C. (2023). Direct preference optimization: Your language model is secretly a reward model. In *Advances in Neural Information Processing Systems 36*. <https://arxiv.org/abs/2305.18290>

[8] Gao, L., Schulman, J., & Hilton, J. (2023). Scaling laws for reward model overoptimization. In *Proceedings of the 40th International Conference on Machine Learning, PMLR 202*. <https://arxiv.org/abs/2210.10760>

[9] Sharma, M., Tong, M., Korbak, T., et al. (2024). Towards understanding sycophancy in language models. In *International Conference on Learning Representations (ICLR)*. <https://arxiv.org/abs/2310.13548>

[10] Irving, G., Christiano, P., & Amodei, D. (2018). AI safety via debate. arXiv preprint arXiv:1805.00899. <https://arxiv.org/abs/1805.00899>

[11] Bowman, S. R., Hyun, J., Perez, E., et al. (2022). Measuring progress on scalable oversight for large language models. arXiv preprint arXiv:2211.03540. <https://arxiv.org/abs/2211.03540>

[12] Burns, C., Izmailov, P., Kirchner, J. H., et al. (2024). Weak-to-strong generalization: Eliciting strong capabilities with weak supervision. In *Proceedings of the 41st International Conference on Machine Learning, PMLR 235*. <https://arxiv.org/abs/2312.09390>